In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
import traceback
import numpy as np
from runner import *
torch.set_printoptions(profile='full')

In [3]:
from easydict import EasyDict as edict
target_config_path = 'config/simple_mlp_cifar.yaml'

try:
    from runner import *
    from utils.logger import setup_logging
    from utils.arg_helper import get_config
except ImportError as e:
    raise e

args = edict()
args.config_file = target_config_path
args.test = False
args.log_level = 'INFO'
args.comment = 'notebook_run'


sample_id=1
config = get_config(args.config_file, sample_id="{:03d}".format(sample_id))

np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
config.use_gpu = config.use_gpu and torch.cuda.is_available()


log_file = os.path.join(config.save_dir, "log_exp_{}.txt".format(config.run_id))


logger = setup_logging(args.log_level, log_file)

logger.info("Writing log file to {}".format(log_file))
logger.info("Exp instance id = {}".format(config.run_id))
logger.info("Exp comment = {}".format(args.comment))

print(f"🚀 正在启动 Runner: {config.runner}")
print(f"📂 结果将保存到: {config.save_dir}")


try:
    runner = eval(config.runner)(config)

    if not args.test:
        # --- [Step 1] Pretrain ---
        print("\n--- [Step 1] Pretraining (InnerNet) ---")
        
        num_cells = config.model.num_cell_types
        print(f"💡 检测到配置需要预训练 {num_cells} 种细胞类型...")

        for i in range(num_cells):
            print(f"   > 正在预训练细胞类型: {i}")
            runner.pretrain(i)

        # --- [Step 2] Phase 1 ---
        print("\n--- [Step 2] Training Phase 1 (Joint Training) ---")
        # runner.train_phase1_v2()

        # # --- [Step 3] Phase 2 ---
        # print("\n--- [Step 3] Training Phase 2 (Freeze Activation) ---")
        # runner.train_phase2()

        # # --- [Step 4] Testing ---
        # print("\n--- [Step 4] Final Testing ---")
        # runner.test()
        
    else:
        # --- Local Test ---
        print("\n--- Local Testing Mode ---")
        runner.test_local()

    print("\n所有任务执行完毕！")

except Exception:
    logger.error(traceback.format_exc())
    print("\n运行出错，错误堆栈如下：")
    traceback.print_exc()

INFO  | 2025-12-24 20:49:20,701 | 2649547340.py             | line 32   : Writing log file to /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar/XorNeuronMLP_v3_001_cifar10_2025-Dec-24-20-49-20/log_exp_846033.txt
INFO  | 2025-12-24 20:49:20,701 | 2649547340.py             | line 33   : Exp instance id = 846033
INFO  | 2025-12-24 20:49:20,702 | 2649547340.py             | line 34   : Exp comment = notebook_run


🚀 正在启动 Runner: XorNeuronRunner_v2
📂 结果将保存到: /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar/XorNeuronMLP_v3_001_cifar10_2025-Dec-24-20-49-20

--- [Step 1] Pretraining (InnerNet) ---
💡 检测到配置需要预训练 2 种细胞类型...
   > 正在预训练细胞类型: 0
Pretrain Loss @ epoch 0001 = 0.025937557808186823
Pretrain Loss @ epoch 0002 = 0.001764909238543235
Pretrain Loss @ epoch 0003 = 0.0015950973091411937
Pretrain Loss @ epoch 0004 = 0.0015746073370596781
Pretrain Loss @ epoch 0005 = 0.0015109011892756154
Pretrain Loss @ epoch 0008 = 0.0014587139576002141
Pretrain Loss @ epoch 0009 = 0.0014318896305534844
Pretrain Loss @ epoch 0011 = 0.0013932382209832302
Pretrain Loss @ epoch 0014 = 0.0013317702934233522
Pretrain Loss @ epoch 0015 = 0.0012700041118575243
Pretrain Loss @ epoch 0016 = 0.001254471473481158
Pretrain Loss @ epoch 0017 = 0.001225820311913636
Pretrain Loss @ epoch 0018 = 0.0012077408864217113
Pretrain Loss @ epoch 0020 = 0.001115388904491358
Pretrain Loss @ epoch 0025 = 0.001045558757309178
Pretrain L

KeyboardInterrupt: 

In [ ]:

# print("\n--- [Step 2] Training Phase 1 (Joint Training) ---")
# runner.train_phase1_v2()


# print("\n--- [Step 3] Training Phase 2 (Freeze Activation) ---")
# runner.train_phase2()

# --- [Step 4] Testing ---
print("\n--- [Step 4] Final Testing ---")
runner.test()


--- [Step 4] Final Testing ---
InnerNetData
train
Files already downloaded and verified


/home/yc/repo/xor_neuron_yoon/utils/train_helper.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_snapshot = torch.load(file_name, map_location=device)


/home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar/XorNeuronMLP_v3_001_cifar10_2025-Dec-24-20-49-20
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_cifar/XorNeuronMLP_v3_001_cifar10_2025-Dec-24-20-49-20/model_snapshot_best_phase2.pth to cuda


100%|██████████| 100/100 [00:00<00:00, 114.62it/s]
INFO  | 2025-12-24 20:50:44,382 | inference_runner.py       | line 506  : Test Accuracy = 0.3255 +- 0


InnerNetData
train
Files already downloaded and verified


NameError: name 'XorNeuronMLP_v3_test' is not defined